# --------------------------------------Introduction--------------------------------------
## XGBoost (Extreme Gradient Boosting):
### What it is:
* `XGBoost(Extreme Gradient Boosting)` is a machine learning algorithm — specifically, it's an implementation of gradient boosting, and it's one of the most widely used models for `tabular` data (like Melbourne housing dataset) in both real-world use and Kaggle competitions. `Builds decision trees sequentially (one at a time)`, unlike Random Forest which `builds trees independently/in parallel`.

### How it works:

1. Start with a `baseline prediction (e.g. average price)`
2. Tree 1 is trained to `predict the error (baseline vs actual)`
3. Add Tree 1's correction to the `baseline → new prediction, new (smaller) error`
4. Tree 2 is trained to predict that `remaining error → add it → error shrinks again`
5. Repeat for many trees, each one `correcting what's still wrong`
6. Final prediction = `baseline + sum of all trees corrections (sum, not average)`

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split

data=pd.read_csv("melb_data.csv")

cols_to_use = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']

X=data[cols_to_use]

y=data.Price

train_X,valid_X,train_y,valid_y=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=0)

In [27]:
from xgboost import XGBRegressor
def calculate_error(model,valid_X,valid_y):
    prediction=model.predict(valid_X)
    error=mean_absolute_error(valid_y,prediction)
    return error

In [28]:
from sklearn.metrics import mean_absolute_error
model1=XGBRegressor()
model1.fit(train_X,train_y)
print("Mean Absolute Error: ",calculate_error(model1,valid_X,valid_y))

Mean Absolute Error:  241657.32839538844


# Parameter Tuning:
### XGBoost has a few parameters that can dramatically affect accuracy and training speed.
#### 1. n_estimators:

`n_estimators` specifies how many times to go through the modeling cycle described above. It is equal to the number of models that we include in the ensemble.

* Too low a value causes underfitting, which leads to inaccurate predictions on both training data and test data.
* Too high a value causes overfitting, which causes accurate predictions on training data, but inaccurate predictions on test data (which is what we care about).

In [29]:
model2=XGBRegressor(n_estimators=500)
model2.fit(train_X,train_y)
print("Mean Absolute Error: ",calculate_error(model2,valid_X,valid_y))

Mean Absolute Error:  250963.21331059


#### 2.early_stopping_rounds:
* `early_stopping_round` offers a way to automatically find the ideal value for `n_estimators`. Early stopping causes the model to stop iterating when the validation score stops improving. It's smart to set a high value for `n_estimators` and then use `early_stopping_rounds` to find the optimal time to stop iterating.
* Setting early_stopping_rounds=5 is a reasonable choice.
* When using `early_stopping_rounds`, we also need to set aside some data for calculating the validation scores - this is done by setting the `eval_set` parameter.

#### 3.eval_set:
* This is the validation data `early_stopping_rounds` checks against, each round, to decide whether to keep going. `eval_set=[(X_valid, y_valid)]` — `a list of one tuple` `(features, target)` — XGBoost predicts on `X_valid` after each new tree, compares to `y_valid`, and tracks whether error is still going down.

#### 4.verbose:
* print training logs or not
* If `True` (or a number), XGBoost prints each round's `evaluation metric` as it trains — useful for `watching progress live`. verbose=`False` silences that output, just runs `quietly`.

In [30]:
model3=XGBRegressor(n_estimators=500,early_stopping_rounds=5)
model3.fit(train_X,train_y,
          eval_set=[(valid_X,valid_y)],
          verbose=False
         )
print("Mean Absolute Error: ",calculate_error(model3,valid_X,valid_y))

Mean Absolute Error:  243816.64362803756


#### 5.learning_rate:
* Remember the "walking toward the target" analogy — each tree's output gets added to the running prediction. learning_rate shrinks how much of each tree's correction actually gets applied. E.g. if a tree predicts a `$60,000 correction`, but `learning_rate=0.05`, only `$60,000 × 0.05 = $3,000` actually gets added. Why shrink it? Taking small, careful steps (with more trees to compensate) generalizes better than taking big, `confident steps` that might `overshoot and overfit` to the training data's noise. `Lower learning rate + more trees = usually more accurate but slower to train.`
##### Imagine we're trying to park a car exactly at a wall, using only "push the gas, coast, see how close we are, repeat."
* High learning rate = each push is a hard stomp on the gas. we might get close fast, but we also risk overshooting past the wall or slamming into it.
* Low learning rate = each push is a very gentle tap. we need many more taps to get there, but we're far less likely to overshoot — we can always correct with the next tiny tap.

#### 6.n_jobs:
* How many CPU cores to use simultaneously while training. `n_jobs=4` means `use 4 cores`. Purely a speed/performance setting — doesn't affect model accuracy or results at all, just how fast it trains.

In [39]:
model4=XGBRegressor(n_estimators=1000,early_stopping_rounds=5,learning_rate=0.05,n_jobs=4)
model4.fit(train_X,train_y,
         eval_set=[(valid_X,valid_y)],
          verbose=False)
print("Mean Absolute Error: ",calculate_error(model4,valid_X,valid_y))

Mean Absolute Error:  240560.49875736376


In [38]:
print(model4.best_iteration)

205
